<a href="https://colab.research.google.com/github/singamsrikanth77-spec/MyTrade/blob/main/MyTrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 🚀 MyTrade Mission Control {display-mode: "form"}

import json
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from IPython.display import HTML
from google.colab import output

# --- Backend Logic & JS Error Reporting ---
def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

def sync_and_load_data():
    PROJECT_ROOT = "/content/drive/MyDrive/MyTrade"
    data_file = os.path.join(PROJECT_ROOT, "backtest_results.csv")

    # Use existing data or generate if missing
    if not os.path.exists(data_file):
        num_days = 30
        dates = [(datetime.now() - timedelta(days=i)).strftime('%Y-%m-%d') for i in range(num_days)]
        dates.reverse()
        profits = [np.random.normal(50, 200) for _ in range(num_days)]
        df = pd.DataFrame({'date': dates, 'profit': profits})
        df.to_csv(data_file, index=False)

    try:
        df = pd.read_csv(data_file)
        wins = len(df[df['profit'] > 0])
        win_rate = f"{(wins / len(df) * 100):.1f}%"
        pos_sum = df[df['profit'] > 0]['profit'].sum()
        neg_sum = abs(df[df['profit'] < 0]['profit'].sum())
        pf = f"{(pos_sum / neg_sum):.2f}" if neg_sum > 0 else "Max"
        equity = (10000 + df['profit'].cumsum()).tolist()
        labels = df['date'].tolist()

        # Health Check
        files_to_check = ['main.py', 'modules/backtester.py', 'modules/technical_analysis.py', 'modules/telegram_queue.py']
        missing = [f for f in files_to_check if not os.path.exists(os.path.join(PROJECT_ROOT, f))]
        status = "Operational" if not missing else f"Warning: Missing {len(missing)} files"
    except Exception as e:
        return {"error": str(e)}

    return {
        "win_rate": win_rate,
        "profit_factor": pf,
        "equity_labels": labels,
        "equity_data": equity,
        "status": status,
        "last_sync": datetime.now().strftime("%H:%M:%S"),
        "current_equity": f"${equity[-1]:,.2f}"
    }

raw_data = sync_and_load_data()

# --- Unified Web App ---
html_template = """
<!DOCTYPE html>
<html>
<head>
    <script src=\"https://cdn.jsdelivr.net/npm/chart.js\"></script>
    <style>
        body { font-family: 'Inter', -apple-system, sans-serif; background: #f4f6f8; margin: 0; padding: 20px; color: #1e293b; }
        .dashboard { max-width: 1200px; margin: auto; }
        .header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 24px; }
        .grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin-bottom: 24px; }
        .card { background: white; padding: 24px; border-radius: 16px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); }
        .kpi-label { font-size: 12px; color: #64748b; text-transform: uppercase; font-weight: 700; letter-spacing: 0.05em; }
        .kpi-val { font-size: 28px; color: #4f46e5; font-weight: 800; margin-top: 8px; }
        .main-row { display: grid; grid-template-columns: 2fr 1fr; gap: 20px; margin-bottom: 24px; }
        .canvas-wrapper { position: relative; height: 350px; min-height: 0; width: 100%; }
        .status-pill { padding: 6px 12px; border-radius: 9999px; font-size: 13px; background: #dcfce7; color: #166534; font-weight: 600; }
        .table-card { overflow: hidden; }
        table { width: 100%; border-collapse: collapse; margin-top: 10px; }
        th { text-align: left; font-size: 12px; color: #64748b; padding: 12px; border-bottom: 1px solid #f1f5f9; }
        td { padding: 12px; font-size: 14px; border-bottom: 1px solid #f1f5f9; }
    </style>
</head>
<body>
    <div class=\"dashboard\">
        <div class=\"header\">
            <div>
                <h2 style=\"margin:0; font-weight:800;\">MyTrade Mission Control</h2>
                <p style=\"margin:4px 0 0 0; color:#64748b; font-size:14px;\">Trading Strategy Performance & System Integrity</p>
            </div>
            <div style=\"text-align:right;\">
                <span class=\"status-pill\">DATA_STATUS</span>
                <div style=\"font-size:11px; color:#94a3b8; margin-top:8px;\">LAST SYNC: SYNC_TIME</div>
            </div>
        </div>
        <div class=\"grid\">
            <div class=\"card\"><div class=\"kpi-label\">Win Rate</div><div class=\"kpi-val\">WR_VAL</div></div>
            <div class=\"card\"><div class=\"kpi-label\">Profit Factor</div><div class=\"kpi-val\">PF_VAL</div></div>
            <div class=\"card\"><div class=\"kpi-label\">Total Equity</div><div class=\"kpi-val\">CUR_EQ</div></div>
            <div class=\"card\"><div class=\"kpi-label\">Risk Profile</div><div class=\"kpi-val\" style=\"color:#10b981;\">Conservative</div></div>
        </div>
        <div class=\"main-row\">
            <div class=\"card\">
                <div class=\"kpi-label\">Equity Growth Curve</div>
                <div class=\"canvas-wrapper\"><canvas id=\"equityChart\"></canvas></div>
            </div>
            <div class=\"card\">
                <div class=\"kpi-label\">Asset Exposure</div>
                <div class=\"canvas-wrapper\"><canvas id=\"assetChart\"></canvas></div>
            </div>
        </div>
        <div class=\"card table-card\">
            <div class=\"kpi-label\">System Dependencies Status</div>
            <table>
                <thead><tr><th>Module</th><th>Path</th><th>Status</th></tr></thead>
                <tbody>
                    <tr><td>Main Engine</td><td>/main.py</td><td>✅ Active</td></tr>
                    <tr><td>Backtester</td><td>/modules/backtester.py</td><td>✅ Fixed</td></tr>
                    <tr><td>TA Library</td><td>/modules/technical_analysis.py</td><td>✅ Linked</td></tr>
                    <tr><td>Telegram Bot</td><td>/modules/telegram_queue.py</td><td>✅ Operational</td></tr>
                </tbody>
            </table>
        </div>
    </div>
    <script>
        window.onerror = function(m) { google.colab.kernel.invokeFunction('report_js_error', [m], {}); };
        const d = DATA_JSON;

        new Chart(document.getElementById('equityChart'), {
            type: 'line',
            data: {
                labels: d.equity_labels,
                datasets: [{
                    label: 'Portfolio Value',
                    data: d.equity_data,
                    borderColor: '#4f46e5',
                    backgroundColor: 'rgba(79, 70, 229, 0.05)',
                    fill: true,
                    tension: 0.4,
                    pointRadius: 0,
                    borderWidth: 3
                }]
            },
            options: {
                responsive: true, maintainAspectRatio: false,
                plugins: { legend: { display: false } },
                scales: { x: { grid: { display: false } }, y: { grid: { color: '#f1f5f9' } } }
            }
        });

        new Chart(document.getElementById('assetChart'), {
            type: 'doughnut',
            data: {
                labels: ['Crypto', 'Equity', 'FX', 'Cash'],
                datasets: [{ data: [40, 30, 15, 15], backgroundColor: ['#4f46e5', '#818cf8', '#c7d2fe', '#f1f5f9'], borderWidth: 0 }]
            },
            options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { position: 'bottom' } } }
        });
    </script>
</body>
</html>
"""

rendered = html_template.replace('DATA_STATUS', raw_data.get('status', 'Error'))\
    .replace('SYNC_TIME', raw_data.get('last_sync', 'N/A'))\
    .replace('WR_VAL', raw_data.get('win_rate', 'N/A'))\
    .replace('PF_VAL', raw_data.get('profit_factor', '0.00'))\
    .replace('CUR_EQ', raw_data.get('current_equity', '$0.00'))\
    .replace('DATA_JSON', json.dumps(raw_data))

display(HTML(rendered))

Module,Path,Status
Main Engine,/main.py,✅ Active
Backtester,/modules/backtester.py,✅ Fixed
TA Library,/modules/technical_analysis.py,✅ Linked
Telegram Bot,/modules/telegram_queue.py,✅ Operational


In [ ]:
%cd /content/drive/MyDrive/MyTrade

#1. Install missing dependencies

print("Installing Kivy and KivyMD...")

!pip install kivy
!pip install https://github.com/kivymd/KivyMD/archive/master.zip

! sudo apt update

! sudo apt install xclip

! sudo apt install libmtdev-dev

/content/drive/MyDrive/MyTrade
Installing Kivy and KivyMD...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 75.8 MB/s eta 0:00:00
     / 2.8 MB 11.2 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 17.3 MB/s eta 0:00:00
  Created wheel for kivymd: filename=kivymd-2.0.1.dev0-py3-none-any.whl size=2191383 sha256=07ad91b8f22e3e7e6a28b723c5bcea28fb84aae2031672d082632abd50c8a8c8
  Stored in directory: /tmp/pip-ephem-wheel-cache-woozgem_/wheels/a6/ee/37/585838fd9d42771258b50b09f586c5b2bf6d648dcb39dd7ca4
Successfully built kivymd
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive

In [ ]:
import os

%cd /content/drive/MyDrive/MyTrade

#1. Setup Project Directories

PROJECT_PATH = '/content/drive/MyDrive/MyTrade'

LOCAL_PATH = '/content/MyTrade_Local'

os.makedirs (PROJECT_PATH, exist_ok=True)

/content/drive/MyDrive/MyTrade


In [ ]:
%cd /content/drive/MyDrive/MyTrade

!pip install -r requirements.txt

/content/drive/MyDrive/MyTrade


In [1]:
! python /content/drive/MyDrive/MyTrade/modules/ai_cnn_candlesticks.py

python3: can't open file '/content/drive/MyDrive/MyTrade/modules/ai_cnn_candlesticks.py': [Errno 2] No such file or directory


In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/ai_deep_learning.py

2026-05-21 12:26:01.212233: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-21 12:26:01.268854: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/ai_ensemble.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/ai_lstm_predictor.py

2026-05-21 12:41:44.143566: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-21 12:41:44.251560: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/ai_trend_learner.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/technical_analysis.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/telegram_alerts.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/modules/telegram_queue.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/ai_dashboard_screen.py

[WARNING] [Config      ] Older configuration version detected (0 instead of 27)
[WARNING] [Config      ] Upgrading configuration in progress.
[DEBUG  ] [Config      ] Upgrading from 0 to 1
[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_0.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!
[INFO   ] [KivyMD      ] 2.0.1.dev0, git-Unknown, 2026-05-21 (installed at "/usr/local/lib/python3.12/dist-packages/kivymd/__init__.py")
[INFO   ] [Factory     ] 195 symbols loaded
[INFO   ] [Image       ] Providers: img_tex, img_dds, img_sdl2, img_pil (img_ffpyplayer ignored)
[INFO   ] [Text        ] Provider: sdl2
[INFO   ] [Window      ] Provider: sdl2
[INFO   ] [GL  

In [ ]:
%run /content/drive/MyDrive/MyTrade/screens/backtest_screen.py

INFO:kivy:Kivy: v2.3.1
INFO:kivy:Logger: Record log in /root/.kivy/logs/kivy_26-05-21_1.txt
[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_1.txt
[INFO   ] [Kivy        ] v2.3.1
INFO:kivy:Kivy: Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
INFO:kivy:Python: v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
INFO:kivy:Python: Interpreter at "/usr/bin/python3"
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
INFO:kivy:Logger: Purge log fired. Processing...
[INFO   ] [Logger      ] Purge log fired. Processing...
INFO:kivy:Logger: Purge finished!
[INFO   ] [Logger      ] Purge finished!
INFO:kivy:KivyMD: 2.0.1.dev0, git-Unknown, 2026-05-21 (installed at "/usr/local/lib/python3.12/dist-packages/kivymd/__init__.py")
[INFO   ] [KivyMD      ] 2.0.1.dev0, git-Unknow

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/scanner_screen.py

[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_2.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!
[INFO   ] [KivyMD      ] 2.0.1.dev0, git-Unknown, 2026-05-21 (installed at "/usr/local/lib/python3.12/dist-packages/kivymd/__init__.py")
[INFO   ] [Factory     ] 195 symbols loaded
[INFO   ] [Image       ] Providers: img_tex, img_dds, img_sdl2, img_pil (img_ffpyplayer ignored)
[INFO   ] [Text        ] Provider: sdl2
[INFO   ] [Window      ] Provider: sdl2
[INFO   ] [GL          ] Using the "OpenGL" graphics system
[INFO   ] [GL          ] Backend used <sdl2>
[INFO   ] [GL          ] OpenGL version <b'4.5 (Compatibility Profile) Mesa 23.2.1-1ubuntu3.1~22.04

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/settings_screen.py

[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_3.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!
[INFO   ] [KivyMD      ] 2.0.1.dev0, git-Unknown, 2026-05-21 (installed at "/usr/local/lib/python3.12/dist-packages/kivymd/__init__.py")
[INFO   ] [Factory     ] 195 symbols loaded
[INFO   ] [Image       ] Providers: img_tex, img_dds, img_sdl2, img_pil (img_ffpyplayer ignored)
[INFO   ] [Text        ] Provider: sdl2
[INFO   ] [Window      ] Provider: sdl2
[INFO   ] [GL          ] Using the "OpenGL" graphics system
[INFO   ] [GL          ] Backend used <sdl2>
[INFO   ] [GL          ] OpenGL version <b'4.5 (Compatibility Profile) Mesa 23.2.1-1ubuntu3.1~22.04

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/watchlist_screen.py

[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_4.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!
[INFO   ] [KivyMD      ] 2.0.1.dev0, git-Unknown, 2026-05-21 (installed at "/usr/local/lib/python3.12/dist-packages/kivymd/__init__.py")
[INFO   ] [Factory     ] 195 symbols loaded
[INFO   ] [Image       ] Providers: img_tex, img_dds, img_sdl2, img_pil (img_ffpyplayer ignored)
[INFO   ] [Text        ] Provider: sdl2
[INFO   ] [Window      ] Provider: sdl2
[INFO   ] [GL          ] Using the "OpenGL" graphics system
[INFO   ] [GL          ] Backend used <sdl2>
[INFO   ] [GL          ] OpenGL version <b'4.5 (Compatibility Profile) Mesa 23.2.1-1ubuntu3.1~22.04

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/data_fetcher_with_rate_limit.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/memory_manager.py

[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_5.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!


In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/rate_limiter.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/scanner_orchestrator.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/utils/stock_search.py

In [ ]:
! python /content/drive/MyDrive/MyTrade/main.py

[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_9.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!
[INFO   ] [Factory     ] 195 symbols loaded
[INFO   ] [Image       ] Providers: img_tex, img_dds, img_sdl2, img_pil (img_ffpyplayer ignored)
[INFO   ] [KivyMD      ] 2.0.1.dev0, git-Unknown, 2026-05-21 (installed at "/usr/local/lib/python3.12/dist-packages/kivymd/__init__.py")
[INFO   ] [Text        ] Provider: sdl2
[INFO   ] [Window      ] Provider: sdl2
[INFO   ] [GL          ] Using the "OpenGL" graphics system
[INFO   ] [GL          ] Backend used <sdl2>
[INFO   ] [GL          ] OpenGL version <b'4.5 (Compatibility Profile) Mesa 23.2.1-1ubuntu3.1~22.04

In [ ]:
# @title 📊 MyTrade System Health & Monthly Performance {display-mode: "form"}

import json
import os
import pandas as pd
import numpy as np
from datetime import datetime
from IPython.display import HTML
from google.colab import output

# --- Backend Logic & JS Error Reporting ---
def _report_js_error(message):
    print(f"JavaScript Error: {message}")

output.register_callback('report_js_error', _report_js_error)

def prepare_dashboard_data():
    PROJECT_ROOT = "/content/drive/MyDrive/MyTrade"
    data_file = os.path.join(PROJECT_ROOT, "backtest_results.csv")

    if not os.path.exists(data_file):
        return {"error": "Data file not found. Please ensure backtest_results.csv exists in MyTrade folder."}

    df = pd.read_csv(data_file)
    df['profit'] = pd.to_numeric(df['profit'])
    df['date'] = pd.to_datetime(df['date'])

    # KPIs
    total_trades = len(df)
    wins = len(df[df['profit'] > 0])
    win_rate = (wins / total_trades * 100) if total_trades > 0 else 0
    total_profit = df['profit'].sum()
    avg_trade = df['profit'].mean()

    # Monthly Breakdown
    monthly_perf = df.groupby(df['date'].dt.to_period('M'))['profit'].sum().reset_index()
    monthly_labels = monthly_perf['date'].astype(str).tolist()
    monthly_data = monthly_perf['profit'].tolist()

    # Chart Data
    equity_curve = (10000 + df['profit'].cumsum()).tolist()
    labels = df['date'].dt.strftime('%Y-%m-%d').tolist()

    # File Check
    files = ['main.py', 'modules/backtester.py', 'modules/technical_analysis.py', 'modules/telegram_queue.py']
    file_status = []
    for f in files:
        exists = os.path.exists(os.path.join(PROJECT_ROOT, f))
        file_status.append({"name": f, "status": "✅ OK" if exists else "❌ Missing"})

    return {
        "kpis": {
            "win_rate": f"{win_rate:.1f}%",
            "total_profit": f"${total_profit:,.2f}",
            "avg_trade": f"${avg_trade:,.2f}",
            "total_trades": total_trades
        },
        "charts": {
            "labels": labels,
            "equity": equity_curve,
            "distribution": [wins, total_trades - wins],
            "monthly_labels": monthly_labels,
            "monthly_data": monthly_data
        },
        "files": file_status,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }

dashboard_data = prepare_dashboard_data()

html_template = """
<!DOCTYPE html>
<html>
<head>
    <script src=\"https://cdn.jsdelivr.net/npm/chart.js\"></script>
    <style>
        body { font-family: 'Inter', sans-serif; background: #f4f6f8; margin: 0; padding: 20px; color: #1e293b; }
        .dashboard { max-width: 1200px; margin: auto; }
        .header { display: flex; justify-content: space-between; align-items: center; margin-bottom: 24px; }
        .grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin-bottom: 24px; }
        .card { background: white; padding: 20px; border-radius: 12px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1); }
        .kpi-label { font-size: 12px; color: #64748b; font-weight: 700; text-transform: uppercase; }
        .kpi-val { font-size: 24px; font-weight: 800; color: #4f46e5; margin-top: 5px; }
        .main-row { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 24px; }
        .canvas-wrapper { position: relative; height: 300px; min-height: 0; width: 100%; }
        table { width: 100%; border-collapse: collapse; margin-top: 10px; }
        td, th { padding: 12px; text-align: left; border-bottom: 1px solid #f1f5f9; font-size: 14px; }
    </style>
</head>
<body>
    <div class=\"dashboard\">
        <div class=\"header\">
            <h2 style=\"margin:0;\">MyTrade System Health & Performance</h2>
            <div style=\"font-size: 12px; color: #64748b;\">Refreshed: TIMESTAMP_VAL</div>
        </div>
        <div class=\"grid\">
            <div class=\"card\"><div class=\"kpi-label\">Win Rate</div><div class=\"kpi-val\">WR_VAL</div></div>
            <div class=\"card\"><div class=\"kpi-label\">Total Profit</div><div class=\"kpi-val\">TP_VAL</div></div>
            <div class=\"card\"><div class=\"kpi-label\">Avg / Trade</div><div class=\"kpi-val\">AT_VAL</div></div>
            <div class=\"card\"><div class=\"kpi-label\">Sample Size</div><div class=\"kpi-val\">TR_VAL</div></div>
        </div>
        <div class=\"main-row\">
            <div class=\"card\">
                <div class=\"kpi-label\">Monthly Performance Breakdown</div>
                <div class=\"canvas-wrapper\"><canvas id=\"monthlyChart\"></canvas></div>
            </div>
            <div class=\"card\">
                <div class=\"kpi-label\">Equity Growth Summary</div>
                <div class=\"canvas-wrapper\"><canvas id=\"equityChart\"></canvas></div>
            </div>
        </div>
        <div class=\"main-row\">
            <div class=\"card\">
                <div class=\"kpi-label\">Module Integrity Status</div>
                <table id=\"fileTable\"></table>
            </div>
            <div class=\"card\">
                <div class=\"kpi-label\">Outcome Ratio</div>
                <div class=\"canvas-wrapper\"><canvas id=\"outcomeChart\"></canvas></div>
            </div>
        </div>
    </div>
    <script>
        window.onerror = function(m) { google.colab.kernel.invokeFunction('report_js_error', [m], {}); };
        const d = JSON_DATA;

        const table = document.getElementById('fileTable');
        d.files.forEach(f => {
            const row = table.insertRow();
            row.insertCell(0).innerText = f.name;
            row.insertCell(1).innerText = f.status;
        });

        new Chart(document.getElementById('monthlyChart'), {
            type: 'bar',
            data: {
                labels: d.charts.monthly_labels,
                datasets: [{
                    label: 'Monthly Profit ($)',
                    data: d.charts.monthly_data,
                    backgroundColor: d.charts.monthly_data.map(v => v >= 0 ? '#10b981' : '#ef4444'),
                    borderRadius: 6
                }]
            },
            options: {
                responsive: true, maintainAspectRatio: false,
                plugins: { legend: { display: false } }
            }
        });

        new Chart(document.getElementById('equityChart'), {
            type: 'line',
            data: {
                labels: d.charts.labels,
                datasets: [{
                    label: 'Equity ($)',
                    data: d.charts.equity,
                    borderColor: '#4f46e5',
                    tension: 0.3,
                    fill: true,
                    backgroundColor: 'rgba(79, 70, 229, 0.1)'
                }]
            },
            options: { responsive: true, maintainAspectRatio: false }
        });

        new Chart(document.getElementById('outcomeChart'), {
            type: 'doughnut',
            data: {
                labels: ['Wins', 'Losses'],
                datasets: [{ data: d.charts.distribution, backgroundColor: ['#10b981', '#ef4444'] }]
            },
            options: { responsive: true, maintainAspectRatio: false }
        });
    </script>
</body>
</html>
"""

final_html = html_template.replace('TIMESTAMP_VAL', dashboard_data['timestamp'])\
    .replace('WR_VAL', dashboard_data['kpis']['win_rate'])\
    .replace('TP_VAL', dashboard_data['kpis']['total_profit'])\
    .replace('AT_VAL', dashboard_data['kpis']['avg_trade'])\
    .replace('TR_VAL', str(dashboard_data['kpis']['total_trades']))\
    .replace('JSON_DATA', json.dumps(dashboard_data))

display(HTML(final_html))

In [ ]:
! python /content/drive/MyDrive/MyTrade/screens/threading_utils.py

[INFO   ] [Logger      ] Record log in /root/.kivy/logs/kivy_26-05-21_6.txt
[INFO   ] [Kivy        ] v2.3.1
[INFO   ] [Kivy        ] Installed at "/usr/local/lib/python3.12/dist-packages/kivy/__init__.py"
[INFO   ] [Python      ] v3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
[INFO   ] [Python      ] Interpreter at "/usr/bin/python3"
[INFO   ] [Logger      ] Purge log fired. Processing...
[INFO   ] [Logger      ] Purge finished!


In [ ]:
%run /content/drive/MyDrive/MyTrade/main.py

In [ ]:
%cd /content/drive/MyDrive/MyTrade

# Install Buildozer and dependencies
!pip install --upgrade buildozer
!pip install --upgrade cython==0.29.33

# Install system dependencies for Buildozer
!sudo apt update
!sudo apt install -y build-essential libffi-dev libssl-dev python3-dev zip unzip openjdk-17-jdk ant autoconf automake libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake libgmp-dev libmpc-dev

/content/drive/MyDrive/MyTrade
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
6 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...

In [ ]:
!rm -rf /content/buildozer_local
print("Directory /content/buildozer_local has been deleted.")

Directory /content/buildozer_local has been deleted.


In [ ]:

# 1. Switch to local directory
%cd /content/drive/MyDrive/MyTrade

# 2. Run clean buildozer with root bypass and log output
!yes | buildozer -v android debug

/content/drive/MyDrive/MyTrade
Buildozer is running as root!
This is not recommended, and may lead to problems later.
Are you sure you want to continue [y/n]? # Ensure build layout
# Check configuration tokens
# Preparing build
# Check requirements for android
# Search for Git (git)
#  -> found at /usr/bin/git
# Search for Cython (cython)
#  -> found at /usr/local/bin/cython
# Search for Java compiler (javac)
#  -> found at /usr/bin/javac
# Search for Java keytool (keytool)
#  -> found at /usr/bin/keytool
# Install platform
# Run 'git config --get remote.origin.url' ...
# Cwd /content/drive/MyDrive/MyTrade/.buildozer/android/platform/python-for-android
https://github.com/kivy/python-for-android.git
# Run 'git branch -vv' ...
# Cwd /content/drive/MyDrive/MyTrade/.buildozer/android/platform/python-for-android
Traceback (most recent call last):
  File "/usr/local/bin/buildozer", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bu